**Ch121a | Module 3: Periodic DFT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/Module3_Periodic-DFT/notebooks/04_example_calculations.ipynb)

# Notebook 4: Example Calculations: Si DOS, TiO₂ Band Gap, Graphene Bands, Co-Pc+CO Spin State

---

## Learning Objectives

- Set up and interpret a two-step SCF → DOS calculation for Si
- Understand the PBE band-gap underestimation problem and how DFT+U partially corrects it for TiO₂
- Read and understand line-mode KPOINTS for a graphene band structure calculation
- Compare high-spin and low-spin VASP inputs for a Co-Pc+CO complex
- Visualize schematic DOS and band structure plots with numpy and matplotlib

## 1. Si Density of States (DOS)

Silicon has a diamond-cubic structure (Fd-3m, 8 atoms/cell) and an indirect band gap of ~1.17 eV (experimental). PBE underestimates this to ~0.6 eV.

### Workflow: SCF → DOS

The DOS calculation requires two steps:
1. **SCF**: converge the charge density and save WAVECAR + CHGCAR
2. **Non-SCF DOS**: read the charge density (ICHARG=11), use tetrahedron smearing (ISMEAR=-5), and increase NEDOS

### INCAR — Step 1: SCF

```
# INCAR — Step 1: Self-consistent field (SCF) for Si
SYSTEM  = Si bulk PBE
ISTART  = 0          # start from scratch
ICHARG  = 2          # superposition of atomic charges
ENCUT   = 400        # plane-wave cutoff (eV)
PREC    = Accurate
EDIFF   = 1E-6       # energy convergence (eV)
NSW     = 0          # no ionic relaxation
IBRION  = -1
ISMEAR  = 0          # Gaussian smearing
SIGMA   = 0.05       # smearing width (eV)
ALGO    = Normal
LWAVE   = .TRUE.     # write WAVECAR for DOS step
LCHARG  = .TRUE.     # write CHGCAR
```

### INCAR — Step 2: DOS

```
# INCAR — Step 2: DOS calculation for Si
SYSTEM  = Si bulk PBE DOS
ISTART  = 1          # read WAVECAR from SCF step
ICHARG  = 11         # non-self-consistent, read CHGCAR
ENCUT   = 400
PREC    = Accurate
EDIFF   = 1E-6
NSW     = 0
IBRION  = -1
ISMEAR  = -5         # tetrahedron method (best for DOS)
SIGMA   = 0.05
NEDOS   = 2000       # number of DOS grid points
LORBIT  = 11         # projected (lm-resolved) DOS
EMIN    = -10.0      # DOS energy range min (eV)
EMAX    =  10.0      # DOS energy range max (eV)
LWAVE   = .FALSE.
LCHARG  = .FALSE.
```

**Key tags**: `NEDOS` sets the number of energy grid points (2000 gives smooth curves). `EMIN`/`EMAX` restrict the energy window. `LORBIT=11` writes atom- and lm-resolved PDOS to DOSCAR.

## 2. TiO₂ Band Gap: PBE vs. PBE+U

Rutile TiO₂ has an experimental band gap of ~3.0 eV. Plain PBE gives ~1.8 eV — a severe underestimation. Adding a Hubbard U on the Ti 3d states (DFT+U) improves this significantly.

### Why PBE underestimates the gap

PBE self-interaction error delocalizes the Ti 3d electrons, artificially narrowing the gap. The Hubbard U adds an extra penalty for partial occupation, effectively pushing unoccupied Ti 3d states up in energy.

| Method | Band gap (eV) | Notes |
|--------|--------------|-------|
| Exp. | ~3.0 | Optical measurement |
| PBE | ~1.8 | Underestimated by ~40% |
| PBE+U (U=4 eV) | ~2.5–2.8 | Significant improvement |
| HSE06 hybrid | ~3.0–3.2 | Most accurate, much more expensive |

### INCAR — Plain PBE

```
# INCAR — TiO2 rutile plain PBE (reference, underestimates gap)
SYSTEM  = TiO2 rutile PBE
ISTART  = 0
ICHARG  = 2
ENCUT   = 500
PREC    = Accurate
EDIFF   = 1E-6
NSW     = 0
IBRION  = -1
ISMEAR  = 0
SIGMA   = 0.05
ALGO    = Normal
NEDOS   = 2000
LORBIT  = 11
LWAVE   = .TRUE.
LCHARG  = .TRUE.
```

### INCAR — PBE+U

```
# INCAR — TiO2 rutile GGA+U (PBE+U)
SYSTEM  = TiO2 rutile PBE+U
ISTART  = 0
ICHARG  = 2
ENCUT   = 500        # higher cutoff for O
PREC    = Accurate
EDIFF   = 1E-6
NSW     = 0
IBRION  = -1
ISMEAR  = 0
SIGMA   = 0.05
ALGO    = Normal
# DFT+U parameters (Dudarev formulation, LDAUTYPE=2)
LDAU    = .TRUE.
LDAUTYPE = 2         # Dudarev: only U_eff = U - J matters
LDAUL   = 2 -1       # l=2 (d) for Ti; -1 (none) for O
LDAUU   = 4.0  0.0   # U for Ti d = 4.0 eV
LDAUJ   = 0.0  0.0   # J = 0 (absorbed into U_eff)
LMAXMIX = 4          # needed for d electrons
LDAUPRINT = 2
NEDOS   = 2000
LORBIT  = 11
LWAVE   = .TRUE.
LCHARG  = .TRUE.
```

## 3. Graphene Band Structure and the Dirac Cone

Graphene is a 2D honeycomb lattice of carbon with a linear band crossing at the **K** point of the BZ — the **Dirac cone**. This makes graphene a zero-gap semiconductor with massless Dirac fermions.

### Band structure calculation: SCF → non-SCF on k-path

1. **SCF** with a dense MP mesh (12×12×1) to get converged CHGCAR
2. **Bands** (non-SCF): use line-mode KPOINTS along Γ→M→K→Γ, read CHGCAR (ICHARG=11)

### KPOINTS — line mode (from `tmp/graphene_bands/vasp/KPOINTS_bands`)

```
Graphene band structure: Gamma-M-K-Gamma (line mode)
40
Line_mode
Reciprocal
  0.0000000000  0.0000000000  0.0000000000  ! Gamma
  0.5000000000  0.0000000000  0.0000000000  ! M

  0.5000000000  0.0000000000  0.0000000000  ! M
  0.3333333333  0.3333333333  0.0000000000  ! K

  0.3333333333  0.3333333333  0.0000000000  ! K
  0.0000000000  0.0000000000  0.0000000000  ! Gamma
```

**Line mode**: pairs of k-points define line segments. `40` = number of points per segment. The K point at (1/3, 1/3, 0) is where the Dirac cone sits.

### Physical significance

- **Linear dispersion** near K: $E = \hbar v_F |k|$, with $v_F \approx 10^6$ m/s
- **Zero effective mass**: electrons behave like photons
- **Berry phase**: π — responsible for absence of backscattering
- **Applications**: high-speed transistors, quantum Hall effect, spintronics

## 4. Co-Pc+CO Spin State: High-spin vs. Low-spin

Cobalt phthalocyanine (CoPc) with a bound CO molecule is a classic example of a transition-metal complex where spin state (high-spin S=3/2 vs. low-spin S=1/2) controls reactivity. DFT+U + vdW is required.

### Workflow

1. Run **high-spin** calculation (MAGMOM for Co set to 3.0 μ_B)
2. Run **low-spin** calculation (MAGMOM for Co set to 1.0 μ_B)
3. Compare total energies → lower energy = ground state

### INCAR — High spin

```
# INCAR — CoPc + CO spin-polarized DFT+U geometry optimization
SYSTEM  = CoPc+CO spin-polarized DFT+U
ISTART  = 0
ICHARG  = 2
ENCUT   = 500
PREC    = Accurate
EDIFF   = 1E-5
EDIFFG  = -0.02      # force convergence (eV/Ang)
NSW     = 100        # max ionic steps
IBRION  = 2          # RMM-DIIS ionic relaxation
ISIF    = 2          # relax atoms, fixed cell
POTIM   = 0.3
# Spin polarization
ISPIN   = 2
MAGMOM  = 1*3.0 32*0.0 8*0.0 16*0.0 2*0.0   ! Co high spin; rest 0
# DFT+U on Co d
LDAU    = .TRUE.
LDAUTYPE = 2
LDAUL   = 2  -1  -1  -1   ! l for Co, C, N, H
LDAUU   = 3.5  0.0  0.0  0.0
LDAUJ   = 0.0  0.0  0.0  0.0
LMAXMIX = 4
LDAUPRINT = 2
# vdW-D3 correction (Grimme)
IVDW    = 12
# Electronic
ISMEAR  = 0
SIGMA   = 0.05
ALGO    = Fast
NELM    = 200
LWAVE   = .TRUE.
LCHARG  = .TRUE.
LORBIT  = 11
NEDOS   = 2000
```

### INCAR — Low spin

```
# INCAR — CoPc + CO low-spin (S=0) DFT+U
SYSTEM  = CoPc+CO low-spin DFT+U
ISTART  = 0
ICHARG  = 2
ENCUT   = 500
PREC    = Accurate
EDIFF   = 1E-5
EDIFFG  = -0.02
NSW     = 100
IBRION  = 2
ISIF    = 2
POTIM   = 0.3
# Low spin — start with zero magnetic moment
ISPIN   = 2
MAGMOM  = 1*1.0 32*0.0 8*0.0 16*0.0 2*0.0
LDAU    = .TRUE.
LDAUTYPE = 2
LDAUL   = 2  -1  -1  -1
LDAUU   = 3.5  0.0  0.0  0.0
LDAUJ   = 0.0  0.0  0.0  0.0
LMAXMIX = 4
LDAUPRINT = 2
IVDW    = 12
ISMEAR  = 0
SIGMA   = 0.05
ALGO    = Fast
NELM    = 200
LWAVE   = .TRUE.
LCHARG  = .TRUE.
LORBIT  = 11
NEDOS   = 2000
```

**Key settings**: `ISPIN=2` enables spin-polarized calculation. `MAGMOM` sets initial magnetic moments — crucial for convergence to the correct spin state. `IVDW=12` (D3-BJ) accounts for dispersion in this large molecular complex.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
rng = np.random.default_rng(42)

# Schematic Si-like DOS: valence band + conduction band with a gap
energies = np.linspace(-12, 8, 2000)
sigma = 0.3   # Gaussian broadening

# Valence band: peaks at ~-8, -5, -2 eV
vb_centers = [-8.0, -5.5, -3.0, -1.2]
vb_weights  = [0.8,  1.5,  2.0,  1.2]

# Conduction band: starts at ~0.6 eV gap above VBM
cb_centers = [1.5, 3.0, 4.5, 6.0]
cb_weights  = [1.0, 1.8, 1.2, 0.6]

dos = np.zeros_like(energies)
for c, w in zip(vb_centers + cb_centers, vb_weights + cb_weights):
    dos += w * np.exp(-0.5 * ((energies - c) / sigma)**2)

# Add noise
dos += rng.normal(0, 0.02, len(energies))
dos = np.clip(dos, 0, None)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(energies, dos, where=(energies < 0), alpha=0.5,
                color='steelblue', label='Valence band (occupied)')
ax.fill_between(energies, dos, where=(energies >= 0.6), alpha=0.5,
                color='tomato', label='Conduction band (empty)')
ax.axvline(0, color='k', lw=1.5, ls='--', label='$E_F$ (VBM set to 0)')
ax.annotate('', xy=(1.5, 3.0), xytext=(0.0, 3.0),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
ax.text(0.75, 3.15, 'PBE gap\n~0.6 eV', ha='center', fontsize=10, color='purple')
ax.set_xlabel('Energy (eV)', fontsize=13)
ax.set_ylabel('DOS (arb. units)', fontsize=13)
ax.set_title('Schematic DOS: Si (PBE)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(-12, 8)
plt.tight_layout()
plt.show()
print('Schematic only. Real DOS from VASP DOSCAR has sharper van Hove singularities.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Schematic graphene band structure along Γ→M→K→Γ
# Tight-binding model: E = ±t*|f(k)|, t=2.7 eV

t = 2.7   # hopping parameter (eV)
nk = 60   # points per segment

def graphene_tb(kx, ky):
    '''Tight-binding energy for graphene.'''
    f = (1 + 4*np.cos(np.sqrt(3)/2*kx)*np.cos(ky/2)
           + 4*np.cos(ky/2)**2)
    E = t * np.sqrt(np.abs(f))
    return E

# Path: Γ(0,0) → M(2π/√3/2, 0) → K(2π/√3*2/3, 0) → Γ
a = 2.46   # Å, graphene lattice constant

# Γ → M
kx_GM = np.linspace(0, np.pi/a, nk)
ky_GM = np.zeros(nk)

# M → K
kx_MK = np.linspace(np.pi/a, 4*np.pi/(3*a), nk)
ky_MK = np.zeros(nk)

# K → Γ
kx_KG = np.linspace(4*np.pi/(3*a), 0, nk)
ky_KG = np.linspace(0, 0, nk)

kx_all = np.concatenate([kx_GM, kx_MK, kx_KG])
ky_all = np.concatenate([ky_GM, ky_MK, ky_KG])

E_plus  = np.array([graphene_tb(kx, ky) for kx, ky in zip(kx_all, ky_all)])
E_minus = -E_plus

k_idx = np.arange(len(kx_all))
tick_pos = [0, nk-1, 2*nk-1, 3*nk-1]
tick_lbl = ['Γ', 'M', 'K', 'Γ']

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(k_idx, E_plus,  'royalblue', lw=2, label='π* (conduction)')
ax.plot(k_idx, E_minus, 'tomato',    lw=2, label='π  (valence)')
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lbl, fontsize=13)
for tp in tick_pos:
    ax.axvline(tp, color='gray', lw=0.5)
ax.set_ylabel('Energy (eV)', fontsize=13)
ax.set_title('Schematic graphene band structure\n(tight-binding, t = 2.7 eV)', fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(-9, 9)
plt.tight_layout()
plt.show()
print('Dirac cone visible at K: linear touching of π and π* bands at E = 0.')

## 5. Further Reading

- Hybertsen & Louie, *Phys. Rev. B* **34**, 5390 (1986) — GW for band gaps
- Anisimov et al., *Phys. Rev. B* **44**, 943 (1991) — original LDA+U
- Castro Neto et al., *Rev. Mod. Phys.* **81**, 109 (2009) — graphene review
- [VASP Wiki: DOS](https://www.vasp.at/wiki/index.php/Total_and_partial_DOS)
- [VASP Wiki: Band structure](https://www.vasp.at/wiki/index.php/Band-structure_calculation)
- [vaspkit Band Structure tutorial](https://vaspkit.com/tutorials.html#band-structure)

---
*Ch121a | Caltech | Module 3 — Notebook 4 of 5*